LangGraph is a newer library from LangChain that lets you build agents and workflows as graphs of nodes (LLM calls, tools, retrievers, etc.), rather than just chains. This makes it easier to design complex multi‑step logic.

In [ ]:
!pip install langchain langchain-core langchain-openai langchain-community langgraph IPython

In [ ]:
import requests
from typing import TypedDict, Annotated

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages
from IPython.display import Image, display

# -------------------------------
# 1. Define Tool (WITH docstring!)
# -------------------------------
@tool
def weather_tool(city: str) -> str:
    """Get current weather for a city."""
    response = requests.get(f"https://wttr.in/{city}?format=3")
    return response.text


tools = [weather_tool]

# -------------------------------
# 2. Define State
# -------------------------------
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


# -------------------------------
# 3. Initialize LLM (tool-enabled)
# -------------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools(tools)

# -------------------------------
# 4. Define Nodes
# -------------------------------
def chatbot(state: AgentState):
    return {
        "messages": [
            llm_with_tools.invoke(state["messages"])
        ]
    }

tool_node = ToolNode(tools)

# -------------------------------
# 5. Routing Logic (IMPORTANT)
# -------------------------------
def route_tools(state: AgentState):
    last_message = state["messages"][-1]

    # If LLM wants to call a tool → go to tool node
    if last_message.tool_calls:
        return "tools"

    # Otherwise → finish
    return END


# -------------------------------
# 6. Build Graph
# -------------------------------
graph = StateGraph(AgentState)

graph.add_node("chatbot", chatbot)
graph.add_node("tools", tool_node)

graph.set_entry_point("chatbot")

graph.add_conditional_edges(
    "chatbot",
    route_tools
)

graph.add_edge("tools", "chatbot")

# Compile
app = graph.compile()

#Draw the graph as a PNG
display(Image(app.get_graph().draw_mermaid_png()))

# -------------------------------
# 7. Run
# -------------------------------
response = app.invoke({
    "messages": [HumanMessage(content="What's the weather in Pune?")]
})

# -------------------------------
# 8. Print Final Answer
# -------------------------------
print(response["messages"][-1].content)

In [ ]:
from typing import TypedDict

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langgraph.graph import StateGraph, END
from IPython.display import Image, display
# -----------------------------
# 1. Prepare documents
# -----------------------------
docs = [
    "Python is a programming language widely used for AI.",
    "LangChain enables building applications with LLMs.",
    "RAG combines retrieval with generation for grounded answers."
]

# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = splitter.create_documents(docs)

# -----------------------------
# 2. Build vector store
# -----------------------------
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# ✅ UPDATED import path (important)
vectorstore = FAISS.from_documents(chunks, embeddings)

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# -----------------------------
# 3. Define state schema (FIXED)
# -----------------------------
class RAGState(TypedDict):
    input: str
    context: list
    output: str

# -----------------------------
# 4. Initialize LLM
# -----------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# -----------------------------
# 5. Define graph nodes
# -----------------------------
def retrieve_node(state: RAGState):
    query = state["input"]

    # ✅ NEW API (avoids deprecation)
    docs = retriever.invoke(query)

    return {
        "input": query,      # preserve state
        "context": docs
    }


def generate_node(state: RAGState):
    query = state["input"]
    docs = state["context"]

    context_text = "\n".join([doc.page_content for doc in docs])

    prompt = f"""
Answer the question using ONLY the context below.

Context:
{context_text}

Question: {query}
"""

    response = llm.invoke(prompt)

    return {
        "input": query,      # preserve state
        "context": docs,
        "output": response.content
    }

# -----------------------------
# 6. Build graph
# -----------------------------
graph = StateGraph(RAGState)

graph.add_node("retrieve", retrieve_node)
graph.add_node("generate", generate_node)

graph.set_entry_point("retrieve")

graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)

# -----------------------------
# 7. Compile
# -----------------------------
app = graph.compile()

#Draw the graph as a PNG
display(Image(app.get_graph().draw_mermaid_png()))
# -----------------------------
# 8. Run
# -----------------------------
result = app.invoke({
    "input": "What is RAG?"
})

print(result["output"])